# ARC-AGI-3 Duck Harness: Qwen -> Gemma -> Qwen, exactly two passes

Every execution runs the real Duck Harness end to end. A normal Kaggle run uses all 25 mounted local `environment_files`; an official competition rerun discovers and executes every game exposed by the official gateway. Every execution branch runs real games.

Qwen acts in pass 1; Gemma reviews only that same-game, same-current-run transcript; Qwen receives the review and acts in pass 2. The artifact keeps the higher-scoring real run for each game. No prior transcript, routebook, replay, solved path, hidden label, cross-run state, or cross-game state is read.


In [1]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [2]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [3]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [4]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [5]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [6]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "qwen_pass1_transcript_for_same_game",
        "gemma_review_of_that_same_transcript",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict no-prior contract active")


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [7]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Fixed real-run configuration

Run every discovered game with concurrency 4 and strict no-prior behavior. A dynamic per-game cap keeps both complete passes inside Kaggle's nine-hour GPU runtime limit. No environment can be omitted.


In [8]:
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY

# Optional grafts are constrained to the same current game and current run.
# Banking and transfer are intentionally never installed.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(
        os.environ.get("TAAF_CONTEXT_WINDOW", "32768")
    ),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags
print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    "games_required=all_discovered "
    f"source_per_game_budget={_original_game_budget}"
)


## 7. Real two-pass Duck Harness relay

Normal execution uses the mounted local Arcade; competition execution uses the official gateway. Both paths run every game twice. Gemma is the between-pass reviewer; Qwen executes both environment passes, and only same-current-run, same-game evidence crosses the relay.


In [9]:
# Exactly two Duck Harness environment runs per game:
#   pass 1: Qwen observes and acts
#   relay:  Gemma reviews only that same-game, same-current-run transcript
#   pass 2: Qwen receives Gemma's review and observes/acts in a fresh run
# The final output keeps the higher-scoring Qwen run for each game.
import copy
import re
import shutil
import signal
from urllib.request import Request

FIRST_PASS_DIR = WORKING_DIR / "relay_pass1_qwen"
GEMMA_REVIEW_DIR = WORKING_DIR / "relay_gemma_reviews"
GEMMA_RUNTIME_DIR = WORKING_DIR / "gemma-llamacpp-bin"
FIRST_PASS_DIR.mkdir(parents=True, exist_ok=True)
GEMMA_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

VLLM_PID = WORKING_DIR / "vllm-openai-server.pid"
VLLM_BASE_URL = "http://127.0.0.1:1234/v1"
VLLM_PROCESS = None
QWEN_SERVED_NAME = "vrfai/Qwen3.6-27B-FP8"
GEMMA_SERVED_NAME = "google/gemma-3-27b-it"
QWEN_DATASET_SLUG = "vrfai-qwen3-6-27b-fp8-hf-snapshot"
GEMMA_DATASET_SLUG = "gemma3-llm-cli"
EXPECTED_LOCAL_GAME_COUNT = 25

FORBIDDEN_PRIOR_INPUTS = [
    "historical_transcripts",
    "yesterday_transcripts",
    "routebooks",
    "replays",
    "solved_paths",
    "hidden_labels",
    "cross_run_state",
    "cross_game_state",
]


def _game_key(value):
    """Return the public environment key, never a cross-game lookup key."""
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive Duck Harness game key from {value!r}")
    return match.group(1)


def _transcript_game_key(path):
    return _game_key(path.name.split("-", 1)[0])


def _dataset_path(owner, slug):
    try:
        mapped = json.loads(os.environ.get("TAAF_KAGGLE_INPUT_PATHS", "{}"))
    except Exception:
        mapped = {}
    candidates = []
    if f"{owner}/{slug}" in mapped:
        candidates.append(Path(str(mapped[f"{owner}/{slug}"])))
    candidates.extend(
        [
            Path("/kaggle/input/datasets") / owner / slug,
            Path("/kaggle/input") / slug,
        ]
    )
    existing = next((path for path in candidates if path.exists()), None)
    if existing is None:
        raise FileNotFoundError(
            f"Attached dataset {owner}/{slug} was not mounted; checked {candidates}"
        )
    return existing


def _gemma_runtime_paths():
    root = _dataset_path("kehhill", GEMMA_DATASET_SLUG)
    model = root / "gemma3" / "gemma-3-27b-it-q4_0.gguf"
    source_runtime = root / "llamacpp-bin"
    missing = [
        str(path)
        for path in (
            model,
            source_runtime / "llama-server",
            source_runtime / "libllama.so",
            source_runtime / "libggml-cuda.so",
        )
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            f"Accessible Gemma 3 27B llama.cpp dataset is incomplete: {missing}"
        )

    # Kaggle input datasets are immutable and may strip executable bits. Stage
    # the complete linked runtime in /kaggle/working before launching it.
    if not GEMMA_RUNTIME_DIR.exists():
        shutil.copytree(source_runtime, GEMMA_RUNTIME_DIR)
    server = GEMMA_RUNTIME_DIR / "llama-server"
    server.chmod(server.stat().st_mode | 0o111)
    runtime_missing = [
        str(path)
        for path in (
            server,
            GEMMA_RUNTIME_DIR / "libllama.so",
            GEMMA_RUNTIME_DIR / "libggml-cuda.so",
        )
        if not path.exists()
    ]
    if runtime_missing or not os.access(server, os.X_OK):
        raise RuntimeError(
            "Staged Gemma llama.cpp runtime is not launchable: "
            f"missing={runtime_missing} executable={os.access(server, os.X_OK)}"
        )
    print(
        f"RELAY staged executable Gemma runtime: {server}",
        flush=True,
    )
    return model, server, GEMMA_RUNTIME_DIR


def _server_models():
    with urlopen(VLLM_BASE_URL + "/models", timeout=10) as response:
        payload = json.loads(response.read().decode("utf-8"))
    return [str(item.get("id")) for item in payload.get("data", [])]


def _assert_server(expected):
    model_ids = _server_models()
    if expected not in model_ids:
        raise RuntimeError(f"Expected active model {expected!r}; found {model_ids!r}")
    print(f"RELAY active model verified: {expected}", flush=True)


def _stop_server():
    global VLLM_PROCESS
    if not VLLM_PID.exists():
        VLLM_PROCESS = None
        return
    try:
        pid = int(VLLM_PID.read_text(encoding="utf-8").strip())
    except (OSError, ValueError):
        VLLM_PID.unlink(missing_ok=True)
        return
    if VLLM_PROCESS is not None and VLLM_PROCESS.pid == pid:
        try:
            VLLM_PROCESS.terminate()
            VLLM_PROCESS.wait(timeout=90)
        except subprocess.TimeoutExpired:
            VLLM_PROCESS.kill()
            VLLM_PROCESS.wait(timeout=30)
    else:
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        deadline = time.monotonic() + 90
        while time.monotonic() < deadline:
            try:
                state = (Path("/proc") / str(pid) / "stat").read_text().split()[2]
                if state == "Z":
                    break
                os.kill(pid, 0)
            except (FileNotFoundError, ProcessLookupError):
                break
            time.sleep(2)
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
    VLLM_PID.unlink(missing_ok=True)
    VLLM_PROCESS = None
    time.sleep(5)


def _start_server(model_path, served_name, family):
    global VLLM_PROCESS
    log_path = WORKING_DIR / f"relay-server-{family}.log"
    site = WORKING_DIR / "vllm-site-packages"
    env = os.environ.copy()
    if family == "gemma":
        expected_model, server, library_dir = _gemma_runtime_paths()
        if Path(model_path) != expected_model:
            raise RuntimeError(
                f"Gemma model path changed: expected={expected_model} actual={model_path}"
            )
        env["LD_LIBRARY_PATH"] = (
            str(library_dir) + os.pathsep + env.get("LD_LIBRARY_PATH", "")
        )
        command = [
            str(server),
            "--model",
            str(model_path),
            "--alias",
            served_name,
            "--host",
            "127.0.0.1",
            "--port",
            "1234",
            "--ctx-size",
            str(int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768"))),
            "--parallel",
            "1",
            "--jinja",
        ]
    else:
        env["PYTHONPATH"] = str(site) + os.pathsep + env.get("PYTHONPATH", "")
        command = [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model",
            str(model_path),
            "--served-model-name",
            served_name,
            "--host",
            "127.0.0.1",
            "--port",
            "1234",
            "--tensor-parallel-size",
            "1",
            "--generation-config",
            "vllm",
            "--enable-prefix-caching",
            "--gpu-memory-utilization",
            "0.92",
            "--max-model-len",
            str(int(os.environ.get("TAAF_CONTEXT_WINDOW", "32768"))),
        ]
        command.extend(
            [
                "--enable-auto-tool-choice",
                "--tool-call-parser",
                "qwen3_coder",
                "--reasoning-parser",
                "qwen3",
                "--default-chat-template-kwargs",
                '{"preserve_thinking": true}',
            ]
        )
    handle = log_path.open("w", encoding="utf-8")
    process = subprocess.Popen(
        command,
        env=env,
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    VLLM_PROCESS = process
    VLLM_PID.write_text(str(process.pid), encoding="utf-8")
    deadline = time.monotonic() + 1200
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-16000:]
            raise RuntimeError(
                f"{family} relay server exited with {process.returncode}\n{tail}"
            )
        try:
            _assert_server(served_name)
            return
        except Exception:
            time.sleep(5)
    tail = log_path.read_text(encoding="utf-8", errors="replace")[-16000:]
    raise TimeoutError(f"{family} relay server did not become ready\n{tail}")


def _patch_qwen_globals():
    import inference.agent.tool_agent as tool_agent_module

    tool_agent_module._LOCAL_ANALYZER_MODEL_ID = QWEN_SERVED_NAME
    tool_agent_module._DEFAULT_ANALYZER_MODEL = QWEN_SERVED_NAME
    tool_agent_module._LOCAL_ANALYZER_BASE_URL = VLLM_BASE_URL
    os.environ["LOCAL_ANALYZER_BASE_URL"] = VLLM_BASE_URL
    os.environ["OPENAI_BASE_URL"] = VLLM_BASE_URL
    os.environ["LOCAL_ANALYZER_MODEL_ID"] = QWEN_SERVED_NAME
    os.environ["INFERENCE_ANALYZER_MODEL"] = QWEN_SERVED_NAME


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    print(
        f"REAL OFFICIAL ARCADE: discovered all {len(game_ids)} gateway games",
        flush=True,
    )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_environment_root():
    root = Path(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    )
    game_dirs = sorted(
        path for path in root.iterdir()
        if path.is_dir() and re.fullmatch(r"[A-Za-z0-9]{4}", path.name)
    )
    if len(game_dirs) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Competition environment_files must contain {EXPECTED_LOCAL_GAME_COUNT} games; "
            f"found {len(game_dirs)} under {root}"
        )
    print(
        f"REAL LOCAL ARCADE: root={root} games={len(game_dirs)}",
        flush=True,
    )
    return root


def _offline_games():
    import arc_agi
    import taaf.game_api

    environment_root = _offline_environment_root()
    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(environment_root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(environment_root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Local Arcade must expose {EXPECTED_LOCAL_GAME_COUNT} games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _fresh_games():
    return _competition_games() if TRUE_SUBMISSION else _offline_games()


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle competition gateway did not become ready: {last_error}")


def _gemma_chat(prompt):
    payload = {
        "model": GEMMA_SERVED_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.15,
        "max_tokens": 800,
    }
    request = Request(
        VLLM_BASE_URL + "/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=90) as response:
        message = json.loads(response.read().decode("utf-8"))["choices"][0]["message"]
    content = str(message.get("content") or "").strip()
    if not content:
        raise RuntimeError("Gemma returned an empty same-run handoff.")
    return content


def _review_with_gemma(game_key, transcript):
    # Tail bounding is a context-management operation on this game's current
    # pass-1 transcript; it does not introduce any prior or other-game input.
    transcript_tail = transcript[-60000:]
    prompt = f"""
You are Gemma, the reviewing model in a strict no-prior Duck Harness relay.
This is game {game_key}. You may use ONLY the pass-1 Qwen transcript below,
which was produced moments ago in this same competition run and same game.
Do not import remembered routes, prior submissions, other games, hidden labels,
or external knowledge about this environment.

Review Qwen's observed objects, tested actions, rewards, state changes, failures,
and partial world model. Return a compact handoff to Qwen for a fresh pass-2
Duck Harness run. Include:
- World model
- Goal model
- Action model
- Confirmed findings
- Failed hypotheses or moves to avoid
- Best next-pass plan
Do not emit tool calls and do not claim facts absent from the transcript.

PASS-1 QWEN TRANSCRIPT FOR GAME {game_key}:
{transcript_tail}
""".strip()
    last_error = None
    for attempt in range(1, 3):
        try:
            review = _gemma_chat(prompt)
            print(
                f"RELAY Gemma review complete: game={game_key} "
                f"attempt={attempt} chars={len(review)}",
                flush=True,
            )
            return review
        except Exception as exc:
            last_error = exc
            print(
                f"RELAY Gemma review retry: game={game_key} "
                f"attempt={attempt} error={exc!r}",
                flush=True,
            )
            time.sleep(5 * attempt)
    raise RuntimeError(
        f"Gemma failed to produce a required handoff for {game_key}: {last_error!r}"
    )


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


from inference.agent.tool_agent import ToolAgent


class GemmaHandoffToolAgent(ToolAgent):
    """Qwen Duck Harness analyzer seeded only with Gemma's same-run review."""

    def __init__(self, *, handoff, game_key, **kwargs):
        self._relay_handoff = str(handoff)
        self._relay_game_key = str(game_key)
        super().__init__(**kwargs)
        self._system_prompt += (
            "\n\nThis is pass 2 of exactly two environment runs. Gemma reviewed "
            "Qwen's pass-1 transcript for this same game. Treat the supplied "
            "review as hypotheses, verify it against the fresh current_frame, "
            "and revise it when live evidence disagrees. No prior-run, replay, "
            "routebook, other-game, or hidden-label information is available."
        )

    def _ensure_session(self, state_path):
        previous = self._session_runtime_dir
        super()._ensure_session(state_path)
        if previous != self._session_runtime_dir:
            self._summarized_knowledge.update(
                {
                    "world_model": self._relay_handoff,
                    "recent_findings": (
                        f"Gemma same-run review for game {self._relay_game_key}; "
                        "verify every claim against the fresh pass-2 frame."
                    ),
                    "current_plan": (
                        "Use the Gemma review to avoid repeated failed probes, "
                        "then search and act through the normal Duck Harness tools."
                    ),
                }
            )


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    print("REAL OFFICIAL ARCADE: competition gateway ready", flush=True)
else:
    _offline_environment_root()

# Fresh ephemeral output state guarantees that no earlier notebook artifact is
# visible to the relay. This only clears this current Kaggle run's working dir.
for stale in (WORKING_DIR / "transcripts").glob("*.txt"):
    stale.unlink()

pass1_game_apis = _fresh_games()
RUN_GAME_COUNT = len(pass1_game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No games are available for the mandatory two-pass run.")

# Kaggle requires GPU competition notebooks to finish within nine hours. Target
# eight and a half hours, reserving 110 minutes for wheel/model setup, both model
# swaps, all Gemma handoffs, artifact validation, and shutdown. Divide the remaining
# time across every discovered game and both environment passes at the selected
# benchmark concurrency. The 1,500-second cap exactly preserves the prior scored
# run's local per-game setting whenever the lane count makes it runtime-feasible.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 110 * 60
MAX_PER_GAME_PASS_SECONDS = 1500.0
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY
    / (2.0 * RUN_GAME_COUNT),
)
RUN_PER_GAME_SECONDS = min(
    MAX_PER_GAME_PASS_SECONDS,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS
bm.games = pass1_game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY
print(
    "REAL RUNTIME BUDGET: "
    f"games={RUN_GAME_COUNT} passes=2 concurrency={TARGET_CONCURRENCY} "
    f"per_game_pass_seconds={RUN_PER_GAME_SECONDS:.3f} "
    f"environment_upper_bound_seconds="
    f"{2 * RUN_GAME_COUNT * RUN_PER_GAME_SECONDS / TARGET_CONCURRENCY:.3f}",
    flush=True,
)

relay_manifest = {
    "protocol": "duck_harness_qwen_gemma_qwen_two_environment_passes_v1",
    "environment_passes": 2,
    "pass_1_actor": "qwen",
    "between_pass_reviewer": "gemma-3-27b-it",
    "pass_2_actor": "qwen",
    "handoff_scope": "same_current_run_same_game_only",
    "forbidden_prior_inputs": FORBIDDEN_PRIOR_INPUTS,
    "uses_yesterday_transcripts": False,
    "uses_yesterday_routes": False,
    "uses_cross_game_state": False,
    "duck_harness_inputs": [
        "benchmark_initial.pkl",
        "deploy_target.pkl",
        "GameAPI",
        "HarnessSolver",
        "current_frame",
        "history",
        "transitions",
        "action(actions)",
    ],
}

try:
    # Fail fast on both model runtimes before spending hours on pass 1. Qwen is
    # already live from setup; briefly smoke-test the accessible Gemma 3 27B
    # GGUF/llama.cpp runtime, then restore Qwen for the first environment pass.
    gemma_model_path, _, _ = _gemma_runtime_paths()
    _stop_server()
    _start_server(gemma_model_path, GEMMA_SERVED_NAME, "gemma")
    gemma_smoke = _gemma_chat(
        "Reply with exactly GEMMA_RELAY_READY and no other text."
    )
    if "GEMMA_RELAY_READY" not in gemma_smoke:
        raise RuntimeError(f"Gemma relay smoke test failed: {gemma_smoke!r}")
    print(
        f"RELAY Gemma smoke test real model output: {gemma_smoke}",
        flush=True,
    )
    _stop_server()
    qwen_model_path = _dataset_path("driessmit1", QWEN_DATASET_SLUG)
    _start_server(qwen_model_path, QWEN_SERVED_NAME, "qwen")

    _assert_server(QWEN_SERVED_NAME)
    _patch_qwen_globals()
    print("RELAY PASS 1/2: Qwen executing Duck Harness", flush=True)
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=True,
    )

    pass1_runs = {_game_key(run.game_id): run for run in bm.game_runs}
    pass1_transcripts = {}
    for source in sorted((WORKING_DIR / "transcripts").glob("*.txt")):
        key = _transcript_game_key(source)
        if key in pass1_transcripts:
            raise RuntimeError(f"Duplicate pass-1 transcript for game {key}")
        destination = FIRST_PASS_DIR / source.name
        destination.write_bytes(source.read_bytes())
        pass1_transcripts[key] = destination
    if set(pass1_transcripts) != set(pass1_runs):
        raise RuntimeError(
            "Pass-1 run/transcript mismatch: "
            f"runs={sorted(pass1_runs)} transcripts={sorted(pass1_transcripts)}"
        )
    if len(pass1_runs) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Pass 1 must complete all {RUN_GAME_COUNT} games; found {len(pass1_runs)}"
        )
    for key in sorted(pass1_runs):
        run = pass1_runs[key]
        print(
            "OFFICIAL SCORE "
            f"game={key} pass=1 actor=qwen score={_run_score(run):.6f} "
            f"levels={_run_levels(run)} actions={_run_actions(run)}",
            flush=True,
        )

    relay_manifest["pass1_games"] = sorted(pass1_runs)
    relay_manifest["pass1_transcripts"] = [
        path.name for path in pass1_transcripts.values()
    ]
    (FIRST_PASS_DIR / "manifest.json").write_text(
        json.dumps(relay_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    # Qwen -> Gemma: Gemma sees only each same-game pass-1 transcript.
    _stop_server()
    gemma_model_path, _, _ = _gemma_runtime_paths()
    _start_server(gemma_model_path, GEMMA_SERVED_NAME, "gemma")
    gemma_reviews = {}
    for key in sorted(pass1_transcripts):
        transcript = pass1_transcripts[key].read_text(
            encoding="utf-8", errors="replace"
        )
        review = _review_with_gemma(key, transcript)
        gemma_reviews[key] = review
        (GEMMA_REVIEW_DIR / f"{key}.txt").write_text(
            review + "\n", encoding="utf-8"
        )
    if set(gemma_reviews) != set(pass1_runs):
        raise RuntimeError("Gemma did not review every pass-1 game.")

    # Gemma -> Qwen: restart Qwen and inject each review through Duck Harness's
    # analyzer_factory. Qwen alone executes the second environment run.
    _stop_server()
    qwen_model_path = _dataset_path("driessmit1", QWEN_DATASET_SLUG)
    _start_server(qwen_model_path, QWEN_SERVED_NAME, "qwen")
    _patch_qwen_globals()

    with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
        bm2 = pickle.load(file)
    bm2.job_dir = WORKING_DIR
    bm2.games = _fresh_games()
    if len(bm2.games) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Pass 2 discovered {len(bm2.games)} games; pass 1 discovered "
            f"{RUN_GAME_COUNT}"
        )
    bm2.n_passes = 1
    bm2.game_weights = None
    bm2.solver = copy.deepcopy(bm.solver)
    bm2.solver.concurrency = TARGET_CONCURRENCY
    bm2.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

    def _pass2_analyzer_factory(game, index):
        key = _game_key(
            getattr(game, "env_name", "")
            or getattr(game, "game_id", "")
            or index
        )
        if key not in gemma_reviews:
            raise RuntimeError(
                f"No same-game Gemma handoff for pass-2 Duck Harness game {key}"
            )
        return GemmaHandoffToolAgent(
            handoff=gemma_reviews[key],
            game_key=key,
            model=QWEN_SERVED_NAME,
            timeout=bm2.solver.analyzer_timeout,
            save_request_logs=bm2.solver.save_request_logs,
            base_url=VLLM_BASE_URL,
            provider="vllm",
        )

    bm2.solver.analyzer_factory = _pass2_analyzer_factory
    print(
        "RELAY PASS 2/2: Qwen executing Duck Harness with Gemma same-run handoffs",
        flush=True,
    )
    await bm2.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=True,
    )

    pass2_runs = {_game_key(run.game_id): run for run in bm2.game_runs}
    if set(pass2_runs) != set(pass1_runs):
        raise RuntimeError(
            "Pass-2 game set differs from pass 1: "
            f"pass1={sorted(pass1_runs)} pass2={sorted(pass2_runs)}"
        )
    for key in sorted(pass2_runs):
        run = pass2_runs[key]
        print(
            "OFFICIAL SCORE "
            f"game={key} pass=2 actor=qwen_after_gemma "
            f"score={_run_score(run):.6f} levels={_run_levels(run)} "
            f"actions={_run_actions(run)}",
            flush=True,
        )

    selected_runs = []
    selection_manifest = []
    for key in sorted(pass1_runs):
        first = pass1_runs[key]
        second = pass2_runs[key]
        candidates = [("qwen_pass1", first), ("qwen_pass2_after_gemma", second)]
        selected_label, selected = max(
            candidates,
            key=lambda item: (
                _run_score(item[1]),
                _run_levels(item[1]),
                -_run_actions(item[1]),
                item[0] == "qwen_pass2_after_gemma",
            ),
        )
        selected_runs.append(selected)
        selection_manifest.append(
            {
                "game_key": key,
                "selected": selected_label,
                "selected_score": _run_score(selected),
                "pass1_score": _run_score(first),
                "pass2_score": _run_score(second),
                "pass1_levels": _run_levels(first),
                "pass2_levels": _run_levels(second),
                "pass1_actions": _run_actions(first),
                "pass2_actions": _run_actions(second),
            }
        )
        print(
            "OFFICIAL BEST "
            f"game={key} pass1={_run_score(first):.6f} "
            f"pass2={_run_score(second):.6f} selected={selected_label} "
            f"selected_score={_run_score(selected):.6f}",
            flush=True,
        )

    relay_manifest["pass2_games"] = sorted(pass2_runs)
    relay_manifest["selection"] = selection_manifest
    bm2.game_runs = selected_runs
    bm2.n_passes = 1
    bm = bm2
    pass1_mean = sum(_run_score(run) for run in pass1_runs.values()) / len(pass1_runs)
    pass2_mean = sum(_run_score(run) for run in pass2_runs.values()) / len(pass2_runs)
    selected_mean = sum(_run_score(run) for run in selected_runs) / len(selected_runs)
    relay_manifest["aggregate_scores"] = {
        "pass1_mean": pass1_mean,
        "pass2_mean": pass2_mean,
        "selected_best_per_game_mean": selected_mean,
    }
    print(
        "OFFICIAL AGGREGATE "
        f"games={RUN_GAME_COUNT} environment_executions={RUN_GAME_COUNT * 2} "
        f"pass1_mean={pass1_mean:.6f} pass2_mean={pass2_mean:.6f} "
        f"selected_best_per_game_mean={selected_mean:.6f}",
        flush=True,
    )
    (WORKING_DIR / "relay_manifest.json").write_text(
        json.dumps(relay_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    (WORKING_DIR / "best_of_two_per_game.json").write_text(
        json.dumps(selection_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    # Keep one selected result per public game key in the required root artifact.
    import pandas as pd

    rows = [
        {
            "row_id": f"{run.game_id}_0",
            "game_id": str(run.game_id),
            "end_of_game": _won(run),
            "score": _run_score(run),
        }
        for run in selected_runs
    ]
    if len(rows) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Official artifact must contain {RUN_GAME_COUNT} rows; found {len(rows)}"
        )
    submission_path = WORKING_DIR / "submission.parquet"
    pd.DataFrame(
        rows,
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(submission_path, index=False)
    written = pd.read_parquet(submission_path)
    required_columns = ["row_id", "game_id", "end_of_game", "score"]
    if list(written.columns) != required_columns:
        raise RuntimeError(
            f"Submission columns are invalid: {list(written.columns)}"
        )
    if len(written) != RUN_GAME_COUNT:
        raise RuntimeError(
            f"Submission must contain {RUN_GAME_COUNT} real game rows; "
            f"found {len(written)}"
        )
    if written["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
        raise RuntimeError("Submission contains duplicate real game IDs")
    if written["score"].isna().any():
        raise RuntimeError("Submission contains missing scores")
    print(
        "REAL SCORE ARTIFACT VALIDATED: "
        f"path={submission_path} rows={len(written)} "
        f"score_sum={float(written['score'].sum()):.6f}",
        flush=True,
    )
    print(
        "RELAY COMPLETE: exactly two Duck Harness passes; "
        f"kept {len(rows)} best-per-game rows",
        flush=True,
    )
finally:
    _stop_server()
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [10]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")

In [11]:
# === CLEAR LOCAL SCORE CARD ===
# Keep this as the final cell. It summarizes the completed local/offline run directly from `bm`.
import re
from html import escape

from IPython.display import HTML, display

if TRUE_SUBMISSION:
    print("Official competition rerun: the local estimate card is intentionally hidden.")
elif not getattr(bm, "game_runs", None):
    print("No completed local benchmark data is available yet.")
else:
    from taaf import diagnostics as taaf_diagnostics

    _summary = taaf_diagnostics.run_summary_text(bm)

    def _extract(pattern, cast=str, default=None):
        match = re.search(pattern, _summary, flags=re.MULTILINE)
        if not match:
            return default
        try:
            return cast(match.group(1).strip())
        except Exception:
            return default

    _mean = _extract(r"^mean score:\s*([0-9.]+)\s*$", float, 0.0)
    _median = _extract(r"^median score:\s*([0-9.]+)\s*$", float, 0.0)
    _duration = _extract(r"^duration:\s*(.+?)\s*$", str, "unknown")
    _games = _extract(r"^games:\s*(\d+)\s*$", int, 0)
    _won = _extract(r"^runs:\s*\d+\s*\(won:\s*(\d+)\)\s*$", int, 0)
    _actions = _extract(r"^total actions:\s*(\d+)\s*$", int, 0)
    _tokens = _extract(r"^total tokens:\s*(\d+)\s*$", int, 0)

    _per_game = re.findall(
        r"^\s+\S+:\s+score=([0-9.]+),\s+levels=([0-9.]+)/([0-9.]+),",
        _summary,
        flags=re.MULTILINE,
    )
    _positive = sum(float(score) > 0 for score, _, _ in _per_game)
    _levels_done = sum(float(done) for _, done, _ in _per_game)
    _levels_total = sum(float(total) for _, _, total in _per_game)

    _budget_s = float(globals().get("RUN_PER_GAME_SECONDS", 0.0) or 0.0)
    _budget_label = (
        f"{_budget_s / 60:.1f} min/game"
        if _budget_s > 0
        else "full local budget"
    )
    _level_label = (
        f"{_levels_done:.0f}/{_levels_total:.0f}"
        if _levels_total > 0
        else "unknown"
    )
    _positive_label = (
        f"{_positive}/{len(_per_game)}"
        if _per_game
        else "unknown"
    )

    display(HTML(f"""
    <div style="border:1px solid #6b7280;border-radius:14px;padding:20px 24px;margin:14px 0;max-width:920px;font-family:Arial,sans-serif">
      <div style="font-size:15px;font-weight:800;letter-spacing:.05em">ARC-AGI-3 TWO-PASS LOCAL SCORE</div>
      <div style="font-size:46px;font-weight:850;line-height:1.15;margin-top:8px">{_mean:.2f}<span style="font-size:18px;font-weight:500"> / 100</span></div>
      <div style="font-size:14px;margin-top:4px">Estimated mean score on the local public environments (not the official hidden-environment leaderboard score)</div>
      <hr style="margin:16px 0;border:none;border-top:1px solid #6b7280">
      <table style="border-collapse:collapse;width:100%;font-size:14px;line-height:1.9">
        <tr><td>Median score</td><td><b>{_median:.2f}</b></td><td>Games fully solved</td><td><b>{_won}/{_games}</b></td></tr>
        <tr><td>Games with positive score</td><td><b>{_positive_label}</b></td><td>Levels completed</td><td><b>{_level_label}</b></td></tr>
        <tr><td>Total actions</td><td><b>{_actions:,}</b></td><td>Total generated tokens</td><td><b>{_tokens:,}</b></td></tr>
        <tr><td>Benchmark duration</td><td><b>{escape(_duration)}</b></td><td>Per-pass game budget</td><td><b>{escape(_budget_label)}</b></td></tr>
      </table>
      <div style="font-size:12px;margin-top:14px;opacity:.78">Every discovered local game completed two Duck Harness environment passes. Qwen acted in both passes, Gemma reviewed every same-run pass-1 transcript, and the displayed score uses the higher-scoring pass for each game.</div>
    </div>
    """))
